In [44]:
# STEP 1: Install Dependencies

!pip install langchain langchain-chroma chromadb sentence-transformers transformers accelerate streamlit

In [45]:
!pip install langchain-huggingface

In [46]:
# STEP 2: Login to Hugging Face

import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
login(token=hf_token)

In [47]:
# STEP 3: Import Libraries

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from transformers import pipeline
import re

In [48]:
# STEP 4: Load Multiple Meeting Sample Transcript

transcripts = {
    "team_sync_2026-03-20": """
    [2026-03-20] John: We should launch the product next Friday.
    [2026-03-20] Sarah: I'll prepare the marketing plan.
    """,
    "team_sync_2026-03-21": """
    [2026-03-21] Mike: The API is still unstable.
    [2026-03-21] John: Let's fix the API before launch.
    """
}


In [49]:
# STEP 5: Tagging Chunks (decision/action_item/discussion/unresolved)

classifier = pipeline("zero-shot-classification" , model="facebook/bart-large-mnli")
labels =["decision" , "action_item" , "discussion" , "unresolved"]

def classify(text):
  result = classifier(text , candidate_labels=labels)
  return result["labels"][0]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [50]:
# STEP 6: Prepare Documents & Chunking with Metadata

docs = []

def extract_date(text):
    match = re.search(r"\[(.*?)]", text)
    return match.group(1) if match else None

for meeting_id, text in transcripts.items():
    chunks = text.strip().split("\n")

    for chunk in chunks:
        tag = classify(chunk)
        docs.append(Document(
            page_content=chunk,
            metadata={
                "tag": tag,
                "date": extract_date(chunk),
                "meeting_id": meeting_id
            }
        ))

# Check
for d in docs:
    print(d.page_content, d.metadata)

[2026-03-20] John: We should launch the product next Friday. {'tag': 'decision', 'date': '2026-03-20', 'meeting_id': 'team_sync_2026-03-20'}
    [2026-03-20] Sarah: I'll prepare the marketing plan. {'tag': 'decision', 'date': '2026-03-20', 'meeting_id': 'team_sync_2026-03-20'}
[2026-03-21] Mike: The API is still unstable. {'tag': 'unresolved', 'date': '2026-03-21', 'meeting_id': 'team_sync_2026-03-21'}
    [2026-03-21] John: Let's fix the API before launch. {'tag': 'decision', 'date': '2026-03-21', 'meeting_id': 'team_sync_2026-03-21'}


In [51]:
# STEP 7: Create Embeddings + Chroma DB

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = Chroma.from_documents(docs , embedding)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [53]:
## Phase 2 - Add Time-Aware + Structured Retrieval UI

In [54]:
# STEP 10: Add SMART QUERY UNDERSTANDING

def parse_query(query):
  query = query.lower()

  tag = None
  if "decision" in query:
    tag= "decision"
  elif "task" in query or "action" in query:
    tag = "action_item"
  elif "discussion" in query:
    tag = "discussion"

  time_filter = None
  if "last week" in query:
    time_filter = "last_week"

  return tag , time_filter

In [55]:
# STEP 11: Apply FILTERS before retrieval

from datetime import datetime , timedelta

def filter_docs(docs , tag=None , time_filter=None):
    filtered = docs

    if tag:
        filtered = [d for d in filtered if d.metadata["tag"] == tag]

    if time_filter == "last_week":
        today = datetime.today()
        last_week = today - timedelta(days=7)
        filtered = [
            d for d in filtered
            if d.metadata["date"] and datetime.strptime(d.metadata["date"], "%Y-%m-%d") >= last_week
        ]
    return filtered

In [56]:
# STEP 12: Hybrid Retrieval

def hybrid_search(query, filtered_docs, k=3):
    # Keyword match
    keyword_docs = [
        d for d in filtered_docs
        if any(word in d.page_content.lower() for word in query.lower().split())
    ]

    # Semantic search
    temp_db = Chroma.from_documents(filtered_docs, embedding)
    semantic_docs = temp_db.similarity_search(query, k=k)

    # Merge + deduplicate
    seen = set()
    combined = []

    for d in keyword_docs + semantic_docs:
        if d.page_content not in seen:
            combined.append(d)
            seen.add(d.page_content)

    return combined[:k]

In [57]:
# STEP 12.5: RAG pipeline

def rag_pipeline(query):
    tag, time_filter = parse_query(query)

    filtered_docs = filter_docs(docs, tag, time_filter)

    if not filtered_docs:
        return "I don't know", []

    results = hybrid_search(query, filtered_docs)

    if not results:
        return "I don't know", []

    # Deduplicate
    seen = set()
    unique_results = []
    for r in results:
        if r.page_content not in seen:
            unique_results.append(r)
            seen.add(r.page_content)

    results = unique_results

    context = "\n".join([r.page_content for r in results])

    query_clean = query.strip()

    prompt = f"""
Answer the question based only on the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {query_clean}
Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer, results


In [58]:
# STEP 13: SHOW RETRIEVAL

query = "What decisions were made last week ?"

answer , results = rag_pipeline(query)

print("ANSWER :\n" , answer)
print("\nRETRIEVED CHUNKS:\n")

for r in results:
  print(r.page_content , r.metadata)

ANSWER :
 I don't know

RETRIEVED CHUNKS:

[2026-03-20] John: We should launch the product next Friday. {'tag': 'decision', 'meeting_id': 'team_sync_2026-03-20', 'date': '2026-03-20'}


In [59]:
# STEP 14: UI (Streamlit) & Timeline/Chart

import streamlit as st
from datetime import datetime
import pandas as pd

st.title("Meeting Intelligence RAG")

query = st.text_input("Ask your question")
tag_filter = st.selectbox("Filter by type:", ["all", "decision", "action_item", "discussion"])
date_filter = st.date_input("Filter by date (optional)")

if query:
    filtered_docs = docs

    if tag_filter != "all":
        filtered_docs = [d for d in filtered_docs if d.metadata["tag"] == tag_filter]

    if date_filter:
        filtered_docs = [
            d for d in filtered_docs
            if d.metadata["date"] and datetime.strptime(d.metadata["date"], "%Y-%m-%d") == date_filter
        ]

    answer, result = rag_pipeline(query)

    st.subheader("Answer")
    st.write(answer)

    st.subheader("Retrieved Context")
    for r in result:
        st.write(r.page_content)
        st.write(r.metadata)

    st.subheader("Chunks contributing to answer")
    for i, r in enumerate(result):
        st.write(f"Chunk {i+1}")
        st.write(r.page_content)
        st.write(r.metadata)

    timeline = pd.DataFrame([
        {"date": d.metadata["date"], "tag": d.metadata["tag"], "content": d.page_content}
        for d in docs if d.metadata.get("date")
    ])

    timeline = timeline.sort_values("date")

    st.subheader("Timeline View")
    st.line_chart(pd.crosstab(timeline["date"], timeline["tag"]))

2026-03-25 17:25:01.180 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 17:25:01.183 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 17:25:01.186 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 17:25:01.188 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 17:25:01.190 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 17:25:01.192 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 17:25:01.194 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 17:25:01.197 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [60]:
# STEP 15: Add “Action Items” View , Timeline View , Multi-meeting support

tasks = [d for d in docs if d.metadata["tag"] == "action_item"]

sorted_docs = sorted(docs , key = lambda x : x.metadata["date"])

for t in tasks:
  print("Date:" , t.metadata.get("date"))
  print("Meeting ID:" , t.metadata.get("meeting_id" , "N/A"))
  print("Content:" , t.page_content)
  print("---")

print("Timeline View:")
for d in sorted_docs:
  print(
      d.metadata.get("date"),
      d.metadata.get("meeting_id" , "N/A"),
      d.metadata["tag"],
      d.page_content
  )

Timeline View:
2026-03-20 team_sync_2026-03-20 decision [2026-03-20] John: We should launch the product next Friday.
2026-03-20 team_sync_2026-03-20 decision     [2026-03-20] Sarah: I'll prepare the marketing plan.
2026-03-21 team_sync_2026-03-21 unresolved [2026-03-21] Mike: The API is still unstable.
2026-03-21 team_sync_2026-03-21 decision     [2026-03-21] John: Let's fix the API before launch.
